In [ ]:
# import necessary libraries
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
np.random.seed(42)


In [ ]:
# Defining the data directories
DATA_DIR = Path("../output")
ARTIFACT_DIR = Path("../artifacts")

orders = pd.read_csv(DATA_DIR / "orders.csv")
inventory = pd.read_csv(DATA_DIR / "inventory_events.csv")
vendors = pd.read_csv(DATA_DIR / "vendors.csv")
forecast_df = pd.read_csv(ARTIFACT_DIR / "weekly_forecast_future.csv")

orders.head()



,order_id,order_date,sku_id,region,payment_type,delivery_days,delivery_status,campaign_applied,order_quantity
0,ORD000000104,0,SKU0002,south,COD,4,DELIVERED,False,1
1,ORD000000103,0,SKU0002,south,COD,3,DELIVERED,False,1
2,ORD000000102,0,SKU0002,south,COD,2,DELIVERED,False,1
3,ORD000000101,0,SKU0002,south,COD,1,DELIVERED,False,1
4,ORD000000100,0,SKU0002,north,PREPAID,5,DELIVERED,False,1


In [ ]:
# Creating inventory snapshot
inventory_snapshot = (
    inventory
    .groupby(["sku_id", "warehouse"])
    .agg(current_stock=("quantity", "sum"))
    .reset_index()
)

inventory_snapshot.rename(columns={"warehouse": "warehouse_id"}, inplace=True)
inventory_snapshot


,sku_id,warehouse_id,current_stock
0,SKU0001,WH_east,38
1,SKU0001,WH_north,0
2,SKU0001,WH_south,25
3,SKU0001,WH_west,37
4,SKU0002,WH_east,1
5,SKU0002,WH_north,22
6,SKU0002,WH_south,36
7,SKU0002,WH_west,49
8,SKU0003,WH_east,36
9,SKU0003,WH_north,183


In [ ]:
# Processing vendor lead times
vendors = vendors.copy()

assert "lead_time_days" in vendors.columns, "vendors.csv missing lead_time_days"

vendors["lead_time_weeks"] = (
    vendors["lead_time_days"] / 7
).apply(np.ceil).astype(int)

vendors[["sku_id", "lead_time_days", "lead_time_weeks", "MOQ"]]


,sku_id,lead_time_days,lead_time_weeks,MOQ
0,SKU0001,13,2,50
1,SKU0002,14,2,50
2,SKU0003,25,4,200
3,SKU0004,30,5,50
4,SKU0005,30,5,200


In [ ]:
# Calculating SKU order history in weeks
sku_history = (
    orders
    .groupby("sku_id")["order_date"]
    .nunique()
    .div(7)
    .astype(int)
    .rename("history_weeks")
    .reset_index()
)

sku_history


,sku_id,history_weeks
0,SKU0001,104
1,SKU0002,104
2,SKU0003,104
3,SKU0004,104
4,SKU0005,104


In [ ]:
# Routing SKUs based on order history length
def route_sku(history_weeks: int) -> str:
    if history_weeks >= 60:
        return "SARIMAX"
    elif history_weeks >= 12:
        return "ML_BASELINE"
    else:
        return "NAIVE"


In [ ]:
# Creating SKU mapping with forecast strategies
sku_map = sku_history.copy()
sku_map["forecast_strategy"] = sku_map["history_weeks"].apply(route_sku)

sku_map["sku_status"] = sku_map["forecast_strategy"].map({
    "SARIMAX": "READY",
    "ML_BASELINE": "LIMITED",
    "NAIVE": "NEW"
})

sku_map


,sku_id,history_weeks,forecast_strategy,sku_status
0,SKU0001,104,SARIMAX,READY
1,SKU0002,104,SARIMAX,READY
2,SKU0003,104,SARIMAX,READY
3,SKU0004,104,SARIMAX,READY
4,SKU0005,104,SARIMAX,READY


In [ ]:
# Combining all data into final context
context = (
    inventory_snapshot
    .merge(vendors, on="sku_id", how="left")
    .merge(sku_map, on="sku_id", how="left")
)

# Final inventory position
context["inventory_position"] = context["current_stock"]

# Hard schema guardrails
required_cols = {
    "sku_id",
    "warehouse_id",
    "lead_time_weeks",
    "MOQ",
    "forecast_strategy",
    "inventory_position"
}
missing = required_cols - set(context.columns)
assert not missing, f"Context missing columns: {missing}"

context


,sku_id,warehouse_id,current_stock,vendor_id,lead_time_days,MOQ,unit_cost,lead_time_weeks,history_weeks,forecast_strategy,sku_status,inventory_position
0,SKU0001,WH_east,38,V_KU0001,13,50,224.26,2,104,SARIMAX,READY,38
1,SKU0001,WH_north,0,V_KU0001,13,50,224.26,2,104,SARIMAX,READY,0
2,SKU0001,WH_south,25,V_KU0001,13,50,224.26,2,104,SARIMAX,READY,25
3,SKU0001,WH_west,37,V_KU0001,13,50,224.26,2,104,SARIMAX,READY,37
4,SKU0002,WH_east,1,V_KU0002,14,50,617.01,2,104,SARIMAX,READY,1
5,SKU0002,WH_north,22,V_KU0002,14,50,617.01,2,104,SARIMAX,READY,22
6,SKU0002,WH_south,36,V_KU0002,14,50,617.01,2,104,SARIMAX,READY,36
7,SKU0002,WH_west,49,V_KU0002,14,50,617.01,2,104,SARIMAX,READY,49
8,SKU0003,WH_east,36,V_KU0003,25,200,105.18,4,104,SARIMAX,READY,36
9,SKU0003,WH_north,183,V_KU0003,25,200,105.18,4,104,SARIMAX,READY,183


In [ ]:
# Demand calculation functions
def demand_from_forecast(forecast_df, sku_id, lead_time_weeks):
    # forecast_df contains SKU-level p50/p90 forecasts per week; sum across lead time
    df = forecast_df[forecast_df["sku_id"] == sku_id].head(lead_time_weeks)
    assert len(df) > 0, f"No forecast found for {sku_id}"
    expected = df["p50"].sum()
    safety = df["p90"].sum() - df["p50"].sum()
    # Return SKU-level totals with explicit demand_source label
    return {
        "expected_demand_LT": expected,
        "safety_stock": safety,
        "demand_source": "PRIMARY_SARIMAX"
    }


def demand_from_history(orders, sku_id, lead_time_weeks):
    # Simple history fallback: weekly aggregation from orders
    weekly = (
        orders[orders["sku_id"] == sku_id]
        .groupby(orders["order_date"] // 7)["order_quantity"]
        .sum()
    )
    avg = weekly.mean() if len(weekly) > 0 else 0.0
    return {
        "expected_demand_LT": float(avg) * lead_time_weeks,
        "safety_stock": float(avg) * 0.3 * lead_time_weeks,
        "demand_source": "FALLBACK_HISTORY"
    }


def demand_for_new_sku(lead_time_weeks):
    # Naive fallback for new SKUs
    BASE = 100
    return {
        "expected_demand_LT": float(BASE) * lead_time_weeks,
        "safety_stock": float(BASE) * 0.5 * lead_time_weeks,
        "demand_source": "FALLBACK_NAIVE"
    }


In [ ]:
# Demand resolution based on forecast strategy
def resolve_demand(row):
    # Ensure lead time available
    assert not pd.isna(row["lead_time_weeks"]), f"Missing lead_time_weeks for {row['sku_id']}"

    if row["forecast_strategy"] == "SARIMAX":
        return demand_from_forecast(
            forecast_df,
            row["sku_id"],
            row["lead_time_weeks"]
        )
    elif row["forecast_strategy"] == "ML_BASELINE":
        return demand_from_history(
            orders,
            row["sku_id"],
            row["lead_time_weeks"]
        )
    else:
        return demand_for_new_sku(row["lead_time_weeks"])


In [ ]:
MIN_SHARE = 0.05  # configurable minimum share per active warehouse (5%)
warehouse_shares = {}
orders_warehouse_col = next((c for c in orders.columns if 'warehouse' in c.lower()), None)
if orders_warehouse_col is not None:
    grp = (
        orders
        .groupby(['sku_id', orders_warehouse_col])['order_quantity']
        .sum()
        .reset_index(name='qty')
    )
    for sku, g in grp.groupby('sku_id'):
        total = float(g['qty'].sum())
        if total <= 0:
            continue
        shares = {row[orders_warehouse_col]: float(row['qty']) / total for _, row in g.iterrows()}
        warehouse_shares[sku] = shares

# Fallback: use inventory_snapshot current_stock distribution
for sku, g in inventory_snapshot.groupby('sku_id'):
    if sku in warehouse_shares:
        continue
    g2 = g.copy()
    g2['qty'] = g2['current_stock'].fillna(0).astype(float)
    total = float(g2['qty'].sum())
    if total > 0:
        shares = {row['warehouse_id']: float(row['qty']) / total for _, row in g2.iterrows()}
    else:
        ids = g2['warehouse_id'].tolist()
        if len(ids) == 0:
            shares = {}
        else:
            eq = 1.0 / len(ids)
            shares = {wid: eq for wid in ids}
    warehouse_shares[sku] = shares

# Ensure every SKU in context has an entry; use equal split as last resort
for sku in context['sku_id'].unique():
    if sku not in warehouse_shares:
        wids = context[context['sku_id'] == sku]['warehouse_id'].unique().tolist()
        if not wids:
            warehouse_shares[sku] = {}
        else:
            eq = 1.0 / len(wids)
            warehouse_shares[sku] = {wid: eq for wid in wids}

# Apply MIN_SHARE floor and renormalize per SKU. This guarantees no active
# warehouse receives less than MIN_SHARE of SKU demand while keeping totals
# summing to 1.0 across warehouses.
final_shares_per_sku = {}
for sku, shares in warehouse_shares.items():
    # active warehouses for this SKU in the current context
    wids = context[context['sku_id'] == sku]['warehouse_id'].unique().tolist()
    if not wids:
        final_shares_per_sku[sku] = {}
        continue

    # build base array aligned with wids
    base_shares = []
    for wid in wids:
        base_shares.append(float(shares.get(wid, 0.0)))
    arr = np.array(base_shares, dtype=float)

    # If all zeros (no history), start with equal split
    if arr.sum() == 0.0:
        arr = np.ones_like(arr) / float(len(arr))

    # enforce minimum share floor
    arr = np.maximum(arr, MIN_SHARE)
    # renormalize to sum to 1.0
    arr = arr / arr.sum()

    final_shares_per_sku[sku] = {wid: float(s) for wid, s in zip(wids, arr)}

# Now build decisions per (sku, warehouse) using allocated demand
results = []
errors = []

for idx, row in context.iterrows():
    sku = row['sku_id']
    wid = row['warehouse_id']

    # Resolve SKU-level demand totals (may raise/assert) and preserve source label
    try:
        sku_demand = resolve_demand(row)
    except AssertionError as ae:
        # Missing lead time or forecast: try history fallback then naive
        try:
            sku_demand = demand_from_history(orders, sku, row['lead_time_weeks'])
            sku_demand['demand_source'] = 'FALLBACK_HISTORY'
        except Exception as he:
            sku_demand = demand_for_new_sku(row['lead_time_weeks'])
            sku_demand['demand_source'] = 'FALLBACK_NAIVE'
            errors.append({'sku_id': sku, 'warehouse_id': wid, 'error': str(ae), 'fallback_error': str(he)})
    except Exception as e:
        errors.append({'sku_id': sku, 'warehouse_id': wid, 'error': str(e)})
        sku_demand = demand_for_new_sku(row['lead_time_weeks'])
        sku_demand['demand_source'] = 'ERROR_FALLBACK'

    # Obtain final share for this sku/wid; default equal split if missing
    shares_map = final_shares_per_sku.get(sku, {})
    share = shares_map.get(wid)
    if share is None:
        wids = context[context['sku_id'] == sku]['warehouse_id'].unique().tolist()
        share = 1.0 / len(wids) if len(wids) > 0 else 1.0

    # Allocate SKU totals to warehouse by share
    expected_total = float(sku_demand.get('expected_demand_LT', 0.0) or 0.0)
    safety_total = float(sku_demand.get('safety_stock', 0.0) or 0.0)
    allocated_expected = expected_total * share
    allocated_safety = safety_total * share

    # Compute reorder logic using warehouse-level allocated demand
    reorder_point = allocated_expected + allocated_safety
    inventory_pos = float(row.get('inventory_position', 0.0) or 0.0)
    reorder_required = inventory_pos <= reorder_point

    gap = reorder_point - inventory_pos
    if reorder_required and row.get('MOQ', 0) > 0:
        recommended_qty = max(int(row['MOQ']), int(np.ceil(gap / row['MOQ']) * int(row['MOQ'])))
    else:
        recommended_qty = 0

    results.append({
        'sku_id': sku,
        'warehouse_id': wid,
        'demand_source': sku_demand.get('demand_source', 'UNKNOWN'),
        'inventory_position': inventory_pos,
        # warehouse-level allocated values
        'expected_demand_LT': round(float(allocated_expected), 2),
        'safety_stock': round(float(allocated_safety), 2),
        'reorder_point': round(float(reorder_point), 2),
        'reorder_required': bool(reorder_required),
        'recommended_order_qty': int(recommended_qty),
        'sku_status': row.get('sku_status')
    })

decision_df = pd.DataFrame(results)

# Observability: show first few fallbacks/errors
if errors:
    print('Warnings / fallbacks during demand resolution (sample up to 10):')
    for e in errors[:10]:
        print(e)

# Diagnostic artifact: aggregate allocated demand per SKU and save for monitoring.
# This artifact helps validate that allocation preserves SKU totals and is not
# used in API responses; it's for senior review and monitoring.
try:
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    sku_alloc_debug = (
        decision_df
        .groupby('sku_id')[['expected_demand_LT', 'safety_stock']]
        .sum()
        .reset_index()
        .rename(columns={
            'expected_demand_LT': 'allocated_expected_sum',
            'safety_stock': 'allocated_safety_sum'
        })
    )
    sku_alloc_debug.to_csv(ARTIFACT_DIR / 'sku_demand_allocation_debug.csv', index=False)
except Exception as e:
    print('Failed to write allocation debug artifact:', e)

# Verification checks (not raising in prod): ensure allocations approximately equal SKU totals
try:
    sku_agg = decision_df.groupby('sku_id')['expected_demand_LT'].sum().rename('allocated_sum')
    sku_totals = None
    if 'p50' in forecast_df.columns:
        sku_totals = forecast_df.groupby('sku_id')['p50'].apply(lambda s: s.head(FORECAST_HORIZON).sum()).rename('forecast_p50_sum')
    diag = pd.concat([sku_totals, sku_agg], axis=1).fillna(0.0)
    diag['diff'] = diag['forecast_p50_sum'] - diag['allocated_sum']
    if (diag['diff'].abs() > 1e-6).any():
        print('\nSanity check: some SKU allocated sums differ from forecast totals (may be due to fallbacks).')
        print(diag.head(10))
except Exception:
    pass

# Guarantee: no NaNs in numeric columns
numeric_cols = ['expected_demand_LT', 'safety_stock', 'reorder_point', 'inventory_position']
for c in numeric_cols:
    if decision_df[c].isna().any():
        decision_df[c] = decision_df[c].fillna(0.0)

# Final output ready for JSON serialization
decision_df


,sku_id,warehouse_id,demand_source,inventory_position,expected_demand_LT,safety_stock,reorder_point,reorder_required,recommended_order_qty,sku_status
0,SKU0001,WH_east,PRIMARY_SARIMAX,38.0,324.23,21.61,345.84,True,350,READY
1,SKU0001,WH_north,PRIMARY_SARIMAX,0.0,42.66,2.84,45.51,True,50,READY
2,SKU0001,WH_south,PRIMARY_SARIMAX,25.0,213.31,14.21,227.53,True,250,READY
3,SKU0001,WH_west,PRIMARY_SARIMAX,37.0,315.70,21.04,336.74,True,300,READY
4,SKU0002,WH_east,FALLBACK_HISTORY,1.0,44.59,13.38,57.97,True,100,READY
5,SKU0002,WH_north,FALLBACK_HISTORY,22.0,181.68,54.50,236.18,True,250,READY
6,SKU0002,WH_south,FALLBACK_HISTORY,36.0,297.29,89.19,386.48,True,400,READY
7,SKU0002,WH_west,FALLBACK_HISTORY,49.0,404.65,121.39,526.04,True,500,READY
8,SKU0003,WH_east,FALLBACK_HISTORY,36.0,190.51,57.15,247.67,True,400,READY
9,SKU0003,WH_north,FALLBACK_HISTORY,183.0,968.43,290.53,1258.96,True,1200,READY


In [49]:
import os
# 1️⃣ Sanity checks — these MUST pass

# No NaNs in critical fields
critical_cols = [
    "expected_demand_LT",
    "safety_stock",
    "reorder_point",
    "inventory_position",
    "recommended_order_qty"
]

assert decision_df[critical_cols].notna().all().all(), "NaNs detected in critical columns"

# No negative quantities
assert (decision_df[critical_cols] >= 0).all().all(), "Negative values detected"

# MOQ respected
bad_moq = decision_df[
    (decision_df["recommended_order_qty"] > 0) &
    (decision_df["recommended_order_qty"] % decision_df["recommended_order_qty"].replace(0, 1) != 0)
]
assert bad_moq.empty, "MOQ violation detected"

# 2️⃣ SKU-level demand consistency check
sku_totals = decision_df.groupby("sku_id")["expected_demand_LT"].sum()
assert (sku_totals > 0).all(), "Some SKUs have zero allocated demand"

# 3️⃣ Summary metrics (for reporting)
summary = {
    "total_skus": decision_df["sku_id"].nunique(),
    "total_warehouses": decision_df["warehouse_id"].nunique(),
    "total_reorders": decision_df["reorder_required"].sum(),
    "total_units_ordered": decision_df["recommended_order_qty"].sum(),
    "fallback_rate": (
        decision_df["demand_source"].str.contains("FALLBACK").mean()
    )
}

print("=== REORDER ENGINE SUMMARY ===")
for k, v in summary.items():
    print(f"{k}: {v}")

# 4️⃣ Export artifacts
ARTIFACT_DIR = "../artifacts"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

decision_df.to_csv(
    f"{ARTIFACT_DIR}/reorder_recommendations.csv",
    index=False
)

pd.DataFrame([summary]).to_csv(
    f"{ARTIFACT_DIR}/reorder_engine_metrics.csv",
    index=False
)

print("\nArtifacts written:")
print("- reorder_recommendations.csv")
print("- reorder_engine_metrics.csv")

print("\nENGINE STATUS: ✅ READY FOR API / DASHBOARD")


=== REORDER ENGINE SUMMARY ===
total_skus: 5
total_warehouses: 4
total_reorders: 20
total_units_ordered: 10950
fallback_rate: 0.8

Artifacts written:
- reorder_recommendations.csv
- reorder_engine_metrics.csv

ENGINE STATUS: ✅ READY FOR API / DASHBOARD


In [45]:
sku_summary = (
    decision_df
    .groupby("sku_id")
    .agg(
        total_inventory=("inventory_position", "sum"),
        total_recommended_qty=("recommended_order_qty", "sum"),
        max_risk=("reorder_required", "max"),
        demand_source=("demand_source", "first")
    )
    .reset_index()
)

sku_summary


,sku_id,total_inventory,total_recommended_qty,max_risk,demand_source
0,SKU0001,100.0,950,True,PRIMARY_SARIMAX
1,SKU0002,108.0,1250,True,FALLBACK_HISTORY
2,SKU0003,352.0,2600,True,FALLBACK_HISTORY
3,SKU0004,74.0,3350,True,FALLBACK_HISTORY
4,SKU0005,457.0,2800,True,FALLBACK_HISTORY


In [48]:
decision_df.to_csv("../artifacts/warehouse_reorder_decisions.csv", index=False)
sku_summary.to_csv("../artifacts/sku_reorder_summary.csv", index=False)
